# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para la elaboración de las celdas Markdown.

Material educativo y demostrativo, sujeto a revisión y validación en Microsoft Fabric.

# NB_09 — Similitud semántica con `ai.similarity`
**Serie:** Microsoft Fabric AI Functions con PySpark  
**Modelo:** `text-embedding-ada-002`  
**Endpoint:** integrado de Fabric

**Objetivo:** comparar tickets contra una referencia y comparar pares de textos, ordenar resultados y auditar ambas operaciones. La similitud no es una probabilidad ni garantiza equivalencia lógica.

## 1. Requisitos y consideraciones

- Workspace con capacidad compatible y AI Functions habilitadas por el administrador.
- Notebook PySpark con SynapseML disponible; en Runtime 2.0 consultar la preparación temporal descrita en la [configuración oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration).
- Usar una sesión limpia con endpoint integrado; no configurar claves externas. Lakehouse adjunto solo para persistencia opcional.
- Datos ficticios; no enviar secretos ni datos personales. Los vectores derivados también requieren controles de acceso.

## 2. Configurar el modelo y las librerías
La función utiliza embeddings, no un modelo generativo de respuestas.

In [1]:
from synapse.ml.spark import aifunc

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

from datetime import datetime, timezone
import uuid

MODEL_NAME = "text-embedding-ada-002"

aifunc.default_conf.set_embedding_deployment_name(MODEL_NAME)

print(f"Modelo configurado: {MODEL_NAME}")
from pyspark import StorageLevel


StatementMeta(, 03c1ae54-a0fd-4f77-9ea8-995d0f83140a, 3, Finished, Available, Finished, False)

Modelo configurado: text-embedding-ada-002


## 3. Referencia del método publicado por Microsoft
[Microsoft Learn — Use ai.similarity with PySpark](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/similarity)

### Sintaxis documentada
Comparación contra un valor:
```python
df.ai.similarity(input_col="col1", other="value", output_col="similarity")
```
Comparación por pares:
```python
df.ai.similarity(input_col="col1", other_col="col2", output_col="similarity")
```

| Parámetro | Tipo | Requisito |
|---|---|---|
| `input_col` | str | Obligatorio |
| `other` | str | Alternativa a `other_col`: texto común |
| `other_col` | str | Alternativa a `other`: columna de comparación |
| `output_col` | str | Opcional; nombre automático si se omite |
| `error_col` | str | Opcional; nombre automático si se omite |

Usar exactamente una de las dos alternativas. El resultado incluye una puntuación por fila, con rango de -1 a 1. Los valores cercanos a 1 indican mayor cercanía; 0 indica ausencia de relación según la referencia oficial. No interpretar una negación como garantía de puntuación negativa. Los valores son relativos y útiles para ranking.

Los ejercicios siguientes son propios, adaptados al contexto de soporte tecnológico.


## 4. Crear datos ficticios

In [2]:
data = [
    (1, "No puedo acceder al portal corporativo", "No puedo acceder al portal corporativo"),
    (2, "El inicio de sesión falla", "No consigo entrar al sistema"),
    (3, "El pipeline de ventas falló", "La carga nocturna de ventas no terminó"),
    (4, "Necesito una silla ergonómica", "Restablecer contraseña"),
    (5, "El portal funciona correctamente", "El portal no funciona correctamente"),
    (6, "Olvidé mi contraseña", "Necesito recuperar mis credenciales")
]
df = spark.createDataFrame(data, ["id_ticket", "descripcion", "texto_comparacion"])
REFERENCIA = "Problemas de acceso al portal corporativo"
display(df)

StatementMeta(, 03c1ae54-a0fd-4f77-9ea8-995d0f83140a, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eb9acf09-6829-4633-940e-3738641d8d3d)

## 5. Comparación contra una referencia fija
Materializar una vez y conservar el resultado en caché. La pérdida de caché puede causar reevaluación y consumo adicional.

In [3]:
run_id = str(uuid.uuid4())
resultado = df.ai.similarity(input_col="descripcion", other=REFERENCIA,
    output_col="similarity", error_col="error_similarity").persist(StorageLevel.MEMORY_AND_DISK)
resultado.collect()
resultado.printSchema()

StatementMeta(, 03c1ae54-a0fd-4f77-9ea8-995d0f83140a, 5, Finished, Available, Finished, False)

root
 |-- id_ticket: long (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- texto_comparacion: string (nullable = true)
 |-- similarity: float (nullable = true)
 |-- error_similarity: struct (nullable = true)
 |    |-- response: string (nullable = true)
 |    |-- status: struct (nullable = true)
 |    |    |-- protocolVersion: struct (nullable = true)
 |    |    |    |-- protocol: string (nullable = true)
 |    |    |    |-- major: integer (nullable = false)
 |    |    |    |-- minor: integer (nullable = false)
 |    |    |-- statusCode: integer (nullable = false)
 |    |    |-- reasonPhrase: string (nullable = true)



## 6. Validación y ranking
El umbral 0.80 es ilustrativo y debe calibrarse con ejemplos etiquetados del dominio.

In [4]:
def validate_scores(frame):
    score = F.col("similarity")
    return frame.withColumn("validation_status",
        F.when(F.col("error_similarity").isNotNull(), "ERROR")
        .when(score.isNull(), "MISSING")
        .when(F.isnan(score) | (score < -1.000001) | (score > 1.000001), "INVALID")
        .otherwise("OK"))
validado = validate_scores(resultado)
display(validado.orderBy(F.desc_nulls_last("similarity")))
display(validado.groupBy("validation_status").count())
display(validado.filter((F.col("validation_status") == "OK") & (F.col("similarity") >= 0.80)))

StatementMeta(, 03c1ae54-a0fd-4f77-9ea8-995d0f83140a, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7070dd0a-d188-4ee2-b0ff-2ff9f4b3072b)

SynapseWidget(Synapse.DataFrame, cf9ccb5d-3a07-4f8f-b274-918db88cf33d)

SynapseWidget(Synapse.DataFrame, 96cbd981-d6c2-4dd7-806b-a17f02d00d06)

## 7. Estadísticas y comparación por pares
Cada modalidad tiene estadísticas independientes. La segunda ejecución consume tokens adicionales y está habilitada para completar el ejercicio; cambiar la bandera para omitirla.

In [ ]:
RUN_PAIRWISE = True
stats_fixed = resultado.ai.stats
display(stats_fixed)
resultado_pares = None
stats_pairs = None
if RUN_PAIRWISE:
    resultado_pares = df.ai.similarity(input_col="descripcion", other_col="texto_comparacion",
        output_col="similarity", error_col="error_similarity").persist(StorageLevel.MEMORY_AND_DISK)
    resultado_pares.collect()
    display(validate_scores(resultado_pares).orderBy(F.desc_nulls_last("similarity")))
    stats_pairs = resultado_pares.ai.stats
    display(stats_pairs)

## 8. Estimar el consumo de capacidad
[Facturación oficial de Microsoft](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing), consultada el 25-11-2026: Ada, 3.36 CU-s por 1.000 tokens de entrada.

$$CU_s = I \times 3.36 / 1000$$

Se usa el total de tokens reportado, sin multiplicarlo nuevamente por el número de pares. No se aplican tarifas GPT. El cálculo solo procede si se confirma el modelo y endpoint integrado. Spark se factura aparte. Las estadísticas ausentes no equivalen a cero.

In [ ]:
RATE_INPUT = 3.36
def metric(stats, name):
    value = stats.get(name)
    return None if value is None else int(value)
def read_stats(frame):
    rows = frame.collect()
    if len(rows) != 1:
        raise RuntimeError("Revisar estadísticas: se esperaba una fila por operación independiente.")
    return rows[0].asDict(recursive=True)
def estimate(stats):
    tokens = metric(stats, "input_tokens")
    if tokens is not None and tokens >= 0 and stats.get("model") == MODEL_NAME and stats.get("client_type") == "fabric_llm_endpoint":
        return tokens * RATE_INPUT / 1000
    return None
runs = [(run_id, "ai.similarity", read_stats(stats_fixed))]
if stats_pairs is not None:
    runs.append((str(uuid.uuid4()), "ai.similarity", read_stats(stats_pairs)))
for mode, (_, _, stats) in zip(["referencia_fija", "pares"], runs):
    print(mode, stats, "CU-s estimados:", estimate(stats))

## 9. Construir registro de auditoría
Se conserva el esquema del NB_08 para compatibilidad. Una vista adicional identifica cada modalidad. El total solo se presenta cuando todas las estimaciones están disponibles.

In [ ]:
num_input_rows = df.count()
audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("execution_timestamp_utc", StringType(), False),
    StructField("notebook_name", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("model_configured", StringType(), True),
    StructField("model_reported", StringType(), True),
    StructField("num_input_rows", LongType(), True),
    StructField("num_successful", LongType(), True),
    StructField("num_exceptions", LongType(), True),
    StructField("num_unevaluated", LongType(), True),
    StructField("num_harmful", LongType(), True),
    StructField("input_tokens", LongType(), True),
    StructField("cached_tokens", LongType(), True),
    StructField("output_tokens", LongType(), True),
    StructField("reasoning_tokens", LongType(), True),
    StructField("estimated_cu_seconds", DoubleType(), True),
    StructField("client_type", StringType(), True),
    StructField("execution_status", StringType(), True)
])

records = []
for rid, operation, stats in runs:
    successful, exceptions, unevaluated, harmful = [metric(stats, x) for x in ("num_successful", "num_exceptions", "num_unevaluated", "num_harmful")]
    if any(v is None for v in (successful, exceptions, unevaluated, harmful)):
        status = "UNKNOWN"
    elif exceptions or harmful:
        status = "PARTIAL"
    elif successful == num_input_rows and unevaluated == 0:
        status = "SUCCESS"
    else:
        status = "INCOMPLETE"
    records.append((rid, datetime.now(timezone.utc).isoformat(), "NB_09_AI_SIMILARITY", operation,
        MODEL_NAME, stats.get("model"), num_input_rows, successful, exceptions, unevaluated, harmful,
        metric(stats,"input_tokens"), metric(stats,"cached_tokens"), metric(stats,"output_tokens"),
        metric(stats,"reasoning_tokens"), estimate(stats), stats.get("client_type"), status))
audit_df = spark.createDataFrame(records, audit_schema)
display(audit_df)
modes = spark.createDataFrame([(r[0], mode) for r, mode in zip(runs, ["referencia_fija", "pares"])], ["run_id", "modalidad"])
display(audit_df.join(modes, "run_id"))
estimates = [estimate(s) for _, _, s in runs]
print("CU-s acumulados:", sum(estimates) if all(v is not None for v in estimates) else None)

## 10. Persistencia opcional en Lakehouse

Si deseas almacenar la auditoría en una tabla Delta del Lakehouse adjunto al notebook, puedes habilitar esta celda.

El modo `append` agrega un registro por ejecución. Verifica previamente el Lakehouse y el nombre de tabla.

In [ ]:
# Descomenta para persistir la auditoría en el Lakehouse adjunto.

# TABLE_NAME = "dbo.audit_ai_functions"
#
# (
#     audit_df.write
#     .format("delta")
#     .mode("append")
#     .saveAsTable(TABLE_NAME)
# )
#
# print(f"Auditoría guardada: {TABLE_NAME}")

## 11. Interpretación
Comparar el ranking con criterios humanos. Una contradicción puede obtener una similitud elevada porque los textos comparten tema. No usar la puntuación como detector de contradicciones o autorización automática. Para búsqueda a gran escala considerar embeddings persistidos y búsqueda vectorial, evitando llamadas repetidas.

## 12. Liberar caché

In [ ]:
resultado.unpersist()
if resultado_pares is not None:
    resultado_pares.unpersist()

## 13. Conclusiones y referencias
Se cubren las dos modalidades oficiales y se audita el consumo de cada ejecución. Validar resultados y tarifas en Fabric; contrastar con Capacity Metrics (operación AI Functions, medidor Copilot and AI).

- [Microsoft Learn — ai.similarity](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/similarity)
- [Configuración PySpark](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration)
- [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)
- [Catálogo AI Functions](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/overview)

Fin del catálogo de nueve funciones. Validación local de sintaxis, sin ejecución en Fabric.